Importing necessary libraries

In [1]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms  
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


Matplotlib is building the font cache; this may take a moment.


Labeling images from dataset based on folder names "good" or "bad"; Creating a master table with images and labels; Splitting into train and test data fairly distributed among labels and categories; Adding extra columns to master table which will later serve as "human oracle" label column with values according to simulated error rate and a isqueried column retruning TRUE if the model asked the oracle for help

In [ ]:
# Your notebooks folder is next to your data folder, so we go up one level (..) and into data
DATA_DIR = '../data' 
CATEGORIES = ['sheet_metal', 'vial', 'wallplugs']
MVTEC_SPLITS = ['train', 'validation', 'test_public']

data_records = []

# 1. Scan folders and collect ground truth
for category in CATEGORIES:
    for mvtec_split in MVTEC_SPLITS:
        
        # We only look for the "good" and "bad" subfolders. 
        # This automatically ignores the "ground_truth" folder!
        for condition, label in [('good', 1), ('bad', 0)]:
            folder_path = os.path.join(DATA_DIR, category, mvtec_split, condition)
            
            # Not all folders exist (e.g., 'train' usually doesn't have a 'bad' folder)
            if not os.path.exists(folder_path):
                continue 
                
            for file_name in os.listdir(folder_path):
                if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    
                    data_records.append({
                        'image_path': os.path.join(folder_path, file_name).replace('\\', '/'), # Ensure slashes are correct
                        'category': category,
                        'true_label': label,
                        'original_folder': mvtec_split # Just in case we want to know where it came from
                    })

# Create the master table
df = pd.DataFrame(data_records)

# 2. Create the Stratify Key (ensures fair distribution of category AND good/bad)
df['stratify_key'] = df['category'] + "_" + df['true_label'].astype(str)

# 3. Split the data (80% for our Active Learning pool, 20% for our final benchmark test)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['stratify_key'], random_state=42)

# 4. Mark the split in our main table
df['split'] = 'train' # Default everything to train
df.loc[test_df.index, 'split'] = 'test' # Overwrite the 20% that are test data

# 5. Setup columns for the Simulated Oracle Experiment!
df['is_queried'] = False  # Will turn True when the AI asks for this image
df['oracle_label'] = None # Will be filled by your 20% or 5% error simulator

# Cleanup: drop the temporary stratify key
df = df.drop(columns=['stratify_key'])

# Save to CSV
df.to_csv('thesis_experiment_data.csv', index=False)

# Print out the results to make sure it worked perfectly
print(f"Total valid images found: {len(df)}")
print("\n--- Final Train/Test Split Summary ---")
# This will show you exactly how many images you have to train and test on!
print(df.groupby(['split', 'category', 'true_label']).size())

print("\n--- First 5 rows of our Master Table ---")
display(df.head())